In [0]:
%pip install Faker

dbutils.library.restartPython()

In [0]:
import importlib

import databricks.jobs.run_daily_generator as daily_runner

importlib.reload(daily_runner)

SOURCE_SCHEMA = "ai_analytics.source"
BRONZE_SCHEMA = "ai_analytics.bronze"

In [0]:
from pyspark.sql import functions as F

latest_dates = (
    spark.table(f"{SOURCE_SCHEMA}.generator_daily_counts")
    .groupBy("entity")
    .agg(F.max("activity_date").alias("latest_date"))
    .orderBy("entity")
    .collect()
)

for row in latest_dates:
    print(row["entity"], row["latest_date"])

In [0]:
BRONZE_TABLES = [
    "user_updates",
    "conversations",
    "messages",
    "completions",
    "model_inferences",
    "feedback",
    "errors",
    "subscriptions",
    "purchases",
    "payments",
]

for entity in BRONZE_TABLES:
    count = (
        spark.table(f"{BRONZE_SCHEMA}.{entity}")
        .filter(
            F.col("dt")
            == F.lit("2026-08-23").cast("date")
        )
        .count()
    )

    print(f"{entity}: {count:,}")

In [0]:
production_dry_run = daily_runner.run_daily_generator(
    spark,
    simulation_date=None,
    seed=42,
    source_schema=SOURCE_SCHEMA,
    bronze_schema=BRONZE_SCHEMA,
    dry_run=True,
)

production_dry_run

In [0]:
from datetime import date

assert production_dry_run["simulation_date"] == date(2026, 8, 23)
assert production_dry_run["mode"] == "DRY_RUN"
assert production_dry_run["bronze_write_attempted"] is False
assert production_dry_run["state_update_attempted"] is False
assert production_dry_run["completed"] is True

print("Clean messages:",
      production_dry_run["clean_counts"]["messages"])

print(
    "Bronze physical messages:",
    production_dry_run["bronze_physical_counts"]["messages"]
)

print(
    "Injected issues:",
    production_dry_run["injection_issue_count"]
)

print("\nPASS: production Aug 23 dry run")
print("PASS: no production writes")

In [0]:
from datetime import date

PRODUCTION_DATE = date(2026, 8, 23)

production_write = daily_runner.run_daily_generator(
    spark,
    simulation_date=PRODUCTION_DATE,
    seed=42,
    source_schema="ai_analytics.source",
    bronze_schema="ai_analytics.bronze",
    dry_run=False,
)

production_write

In [0]:
assert production_write["mode"] == "WRITE"
assert production_write["completed"] is True

assert production_write["bronze_write_attempted"] is True
assert production_write["state_update_attempted"] is True

assert (
    production_write["post_write_validation"]["status"]
    == "PASS"
)

print("PASS: production Aug 23 completed")

In [0]:
print(
    "Clean messages:",
    production_write["clean_counts"]["messages"]
)

print(
    "Bronze physical messages:",
    production_write["bronze_physical_counts"]["messages"]
)

print(
    "Injected issues:",
    production_write["injection_issue_count"]
)

print(
    "State update completed:",
    production_write["state_update_summary"]["completed"]
)

In [0]:
from pyspark.sql import functions as F

latest_dates = (
    spark.table("ai_analytics.source.generator_daily_counts")
    .groupBy("entity")
    .agg(F.max("activity_date").alias("latest_date"))
    .orderBy("entity")
    .collect()
)

for row in latest_dates:
    print(row["entity"], row["latest_date"])

In [0]:
bronze_messages = (
    spark.table("ai_analytics.bronze.messages")
    .filter(
        F.col("dt")
        == F.lit("2026-08-23").cast("date")
    )
    .count()
)

clean_message_count = (
    spark.table("ai_analytics.source.generator_daily_counts")
    .filter(
        (F.col("entity") == "messages")
        & (
            F.col("activity_date")
            == F.lit("2026-08-23").cast("date")
        )
    )
    .first()["row_count"]
)

print("Bronze physical messages:", bronze_messages)
print("Clean generator messages:", clean_message_count)

assert bronze_messages == 31318
assert clean_message_count == 31256

print("PASS: dirty Bronze and clean source state remain separated")

In [0]:
retry = daily_runner.run_daily_generator(
    spark,
    simulation_date=PRODUCTION_DATE,
    seed=42,
    source_schema="ai_analytics.source",
    bronze_schema="ai_analytics.bronze",
    dry_run=False,
)

retry

In [0]:
from pyspark.sql import functions as F

print("PRODUCTION SOURCE STATE")

latest_dates = (
    spark.table("ai_analytics.source.generator_daily_counts")
    .groupBy("entity")
    .agg(F.max("activity_date").alias("latest_date"))
    .orderBy("entity")
    .collect()
)

for row in latest_dates:
    print(row["entity"], row["latest_date"])

In [0]:
BRONZE_TABLES = [
    "user_updates",
    "conversations",
    "messages",
    "completions",
    "model_inferences",
    "feedback",
    "errors",
    "subscriptions",
    "purchases",
    "payments",
]

print("\nPRODUCTION AUG 23 BRONZE")

for entity in BRONZE_TABLES:
    count = (
        spark.table(f"ai_analytics.bronze.{entity}")
        .filter(
            F.col("dt") == F.lit("2026-08-23").cast("date")
        )
        .count()
    )

    print(f"{entity}: {count:,}")

In [0]:
from pyspark.sql import functions as F

BRONZE_TABLES = [
    "user_updates",
    "conversations",
    "messages",
    "completions",
    "model_inferences",
    "feedback",
    "errors",
    "subscriptions",
    "purchases",
    "payments",
]

EXPECTED_AUG23 = {
    "user_updates": 41,
    "conversations": 2472,
    "messages": 31318,
    "completions": 30948,
    "model_inferences": 30931,
    "feedback": 2045,
    "errors": 747,
    "subscriptions": 0,
    "purchases": 48,
    "payments": 51,
}

for entity in BRONZE_TABLES:
    actual = (
        spark.table(f"ai_analytics.bronze.{entity}")
        .filter(
            F.col("dt")
            == F.lit("2026-08-23").cast("date")
        )
        .count()
    )

    expected = EXPECTED_AUG23[entity]

    print(
        f"{entity}: "
        f"expected={expected:,}, "
        f"actual={actual:,}, "
        f"{'MATCH' if actual == expected else 'DIFFERENT'}"
    )

    assert actual == expected

print("\nPASS: production Aug 23 Bronze is complete")

In [0]:
aug23_clean_counts = {
    row["entity"]: row["row_count"]
    for row in (
        spark.table(
            "ai_analytics.source.generator_daily_counts"
        )
        .filter(
            F.col("activity_date")
            == F.lit("2026-08-23").cast("date")
        )
        .collect()
    )
}

print(aug23_clean_counts)

assert aug23_clean_counts == {
    "user_updates": 41,
    "conversations": 2472,
    "messages": 31256,
    "errors": 747,
}

print("PASS: production clean generator state is complete")